# 第二篇论文：按第一篇修订稿式 (51) 计算有限效率成功概率

本 notebook 使用第一篇论文修订稿的有限效率 PNRD 模型，计算第二篇论文的 CPS、OCS 与 ECS 构型。两篇论文唯一的物理替换是主 OPA 的 signal 输入态：第一篇为压缩真空，第二篇为相干态或奇/偶猫态；有限效率的概率结构完全相同。

本 notebook 明确区分三种量：

$$
P_{\rm dominant}^{(46)}
=\eta_{\rm prep}^{m}\eta_{\rm out}^{n}
P_{\rm Fock}(m,g')P_{\psi}(g,m,n),
$$

即修订稿式 (46) 的主导正确登记通道 $(j,l)=(m,n)$；

$$
\begin{aligned}
P_{\rm trial}^{(51)}(m,n)
=&\sum_{j=m}^{\infty}\sum_{l=n}^{\infty}
\binom{j}{m}\eta_{\rm prep}^{m}(1-\eta_{\rm prep})^{j-m}\\
&\times\binom{l}{n}\eta_{\rm out}^{n}(1-\eta_{\rm out})^{l-n}
P_{\rm Fock}(j,g')P_{\psi}(g,j,l),
\end{aligned}
$$

即修订稿式 (51) 的完整有限效率注册概率；以及旧第二篇表格所用的单一效率近似

$$
P_{\rm legacy}=\eta_{\rm out}P_{\rm Fock}(m,g')P_{\psi}(g,m,n).
$$

`P_legacy` 仅保留用于说明原 Tables I--IV 数值的来源，**不能作为修订后的成功概率**。用于更新四个表的主结果应为 `P_trial_eq51`；`P_dominant_eq46` 用于展示仅保留正确登记通道时的近似。


## 模型卡

- **信号模式**：单模截断 Fock 空间，维数 `N_DIM`；idler 模式通过第二篇论文式 (3)-(4) 的解析条件态表示。
- **输入态**：CPS 使用相干态；OCS/ECS 使用归一化奇/偶猫态。辅助 SPDC 的实际 Fock 对数为 $j$，概率为 $P_{\rm Fock}(j,g')$。
- **登记事件**：前端 PNRD 登记 $m$ 个光子，主 OPA 输出 PNRD 登记 $n$ 个光子。
- **真实光子数**：$j$ 是实际输入主 OPA 的 idler 光子数；$l$ 是主 OPA 输出端实际光子数。有限效率时一般有 $(j,l)\ne(m,n)$。
- **主结果**：`P_trial_eq51` 是修订稿式 (51) 的有限截断计算；`F_conditional_eta` 是同一登记事件对应的条件混态保真度。
- **正确主导通道**：`P_dominant_eq46` 是 $(j,l)=(m,n)$ 项，含 $\eta_{\rm prep}^{m}\eta_{\rm out}^{n}$。
- **数值假设**：独立光子损失型 PNRD、无暗计数、无串扰；计算完全确定性。


In [1]:
# 1. 导入依赖
from __future__ import annotations

import csv
import math
from pathlib import Path

import numpy as np
import qutip as qt


In [2]:
# 2. EDIT HERE: 物理参数、探测效率和数值截断
# 辅助 SPDC 增益：g_prep=1.0483，对应 tanh(rho_prime)≈0.30。
G_PREP = 1.0483

# 第一篇修订稿式 (46) 与式 (51) 中的两个 PNRD 效率。
# 改这两个数后，必须从头运行 notebook，并使用新生成的 CSV。
ETA_PREP = 0.95   # 辅助 SPDC / Fock 态制备 heralding PNRD: eta'_det
ETA_OUT = 0.95    # 主 OPA 输出端 PNRD: eta_det

# 截断参数。修改后必须比较更大截断，确认 Eq. (51) 的 P_trial 和 F 已收敛。
N_DIM = 50
J_MAX = 8
L_MAX = 14
TOP_CHANNELS_PER_CASE = 0

# 新目录避免覆盖任何旧的 single-eta 或 two-stage CSV。
OUTPUT_DIR = Path("outputs") / "second_paper_eq51_corrected_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 以下 CASES 为原 notebook 的论文参数，保持不变。
CASES = [
    # Table II: odd cat state (OCS), catalytic configurations
    {"label": "Table II OCS (1,1), alpha=0.30", "m": 1, "n": 1, "g": 1.60, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.58, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 1.39e-3, "paper_fidelity": 1.000},
    {"label": "Table II OCS (1,1), alpha=1.06", "m": 1, "n": 1, "g": 1.60, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 2.02, "target_gamma": 0.340, "target_parity": "odd", "paper_p_trial": 2.73e-3, "paper_fidelity": 0.992},
    {"label": "Table II OCS (1,1), alpha=1.31", "m": 1, "n": 1, "g": 1.60, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.31, "target_gamma": 0.390, "target_parity": "odd", "paper_p_trial": 3.94e-3, "paper_fidelity": 0.976},
    {"label": "Table II OCS (2,2), alpha=0.30", "m": 2, "n": 2, "g": 1.36, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.46, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 2.78e-4, "paper_fidelity": 1.000},
    {"label": "Table II OCS (2,2), alpha=1.06", "m": 2, "n": 2, "g": 1.36, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 1.67, "target_gamma": 0.310, "target_parity": "odd", "paper_p_trial": 3.33e-4, "paper_fidelity": 0.999},
    {"label": "Table II OCS (2,2), alpha=1.31", "m": 2, "n": 2, "g": 1.36, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.11, "target_gamma": 0.450, "target_parity": "odd", "paper_p_trial": 3.71e-4, "paper_fidelity": 0.995},
    {"label": "Table II OCS (3,3), alpha=0.30", "m": 3, "n": 3, "g": 4.50, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.68, "target_gamma": 0.050, "target_parity": "odd", "paper_p_trial": 5.73e-7, "paper_fidelity": 1.000},
    {"label": "Table II OCS (3,3), alpha=1.06", "m": 3, "n": 3, "g": 4.50, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 2.30, "target_gamma": 0.430, "target_parity": "odd", "paper_p_trial": 1.66e-6, "paper_fidelity": 0.980},
    {"label": "Table II OCS (3,3), alpha=1.31", "m": 3, "n": 3, "g": 4.50, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.56, "target_gamma": 0.470, "target_parity": "odd", "paper_p_trial": 2.64e-6, "paper_fidelity": 0.958},
    {"label": "Table II OCS (4,4), alpha=0.30", "m": 4, "n": 4, "g": 2.60, "alpha_in": 0.30, "input_parity": "odd", "target_alpha": 0.47, "target_gamma": 0.030, "target_parity": "odd", "paper_p_trial": 2.74e-7, "paper_fidelity": 1.000},
    {"label": "Table II OCS (4,4), alpha=1.06", "m": 4, "n": 4, "g": 2.60, "alpha_in": 1.06, "input_parity": "odd", "target_alpha": 1.72, "target_gamma": 0.320, "target_parity": "odd", "paper_p_trial": 3.41e-7, "paper_fidelity": 0.999},
    {"label": "Table II OCS (4,4), alpha=1.31", "m": 4, "n": 4, "g": 2.60, "alpha_in": 1.31, "input_parity": "odd", "target_alpha": 2.24, "target_gamma": 0.480, "target_parity": "odd", "paper_p_trial": 3.94e-7, "paper_fidelity": 0.998},

    # Additional Table II row supplied after the original table calculation.
    {"label": "Table II OCS (6,6), alpha=1.00", "m": 6, "n": 6, "g": 1.40, "alpha_in": 1.00, "input_parity": "odd", "target_alpha": 1.47, "target_gamma": 0.300, "target_parity": "odd", "paper_fidelity": 1.000},

    # Table III: even cat state (ECS), catalytic configurations
    {"label": "Table III ECS (1,1), alpha=0.50", "m": 1, "n": 1, "g": 1.10, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.37, "target_gamma": 0.015, "target_parity": "even", "paper_p_trial": 4.91e-2, "paper_fidelity": 1.000},
    {"label": "Table III ECS (1,1), alpha=1.00", "m": 1, "n": 1, "g": 1.10, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 0.74, "target_gamma": 0.061, "target_parity": "even", "paper_p_trial": 3.64e-2, "paper_fidelity": 1.000},
    {"label": "Table III ECS (1,1), alpha=1.50", "m": 1, "n": 1, "g": 1.10, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 1.13, "target_gamma": 0.149, "target_parity": "even", "paper_p_trial": 1.67e-2, "paper_fidelity": 0.997},
    {"label": "Table III ECS (2,2), alpha=0.50", "m": 2, "n": 2, "g": 2.80, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.85, "target_gamma": 0.070, "target_parity": "even", "paper_p_trial": 1.62e-5, "paper_fidelity": 1.000},
    {"label": "Table III ECS (2,2), alpha=1.00", "m": 2, "n": 2, "g": 2.80, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.71, "target_gamma": 0.246, "target_parity": "even", "paper_p_trial": 4.43e-5, "paper_fidelity": 0.999},
    {"label": "Table III ECS (2,2), alpha=1.50", "m": 2, "n": 2, "g": 2.80, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.43, "target_gamma": 0.397, "target_parity": "even", "paper_p_trial": 9.49e-5, "paper_fidelity": 0.991},
    {"label": "Table III ECS (3,3), alpha=0.50", "m": 3, "n": 3, "g": 1.93, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.73, "target_gamma": 0.016, "target_parity": "even", "paper_p_trial": 3.33e-6, "paper_fidelity": 1.000},
    {"label": "Table III ECS (3,3), alpha=1.00", "m": 3, "n": 3, "g": 1.93, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.51, "target_gamma": 0.262, "target_parity": "even", "paper_p_trial": 5.70e-6, "paper_fidelity": 0.999},
    {"label": "Table III ECS (3,3), alpha=1.50", "m": 3, "n": 3, "g": 1.93, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.70, "target_gamma": 0.661, "target_parity": "even", "paper_p_trial": 8.60e-6, "paper_fidelity": 0.981},
    {"label": "Table III ECS (4,4), alpha=0.50", "m": 4, "n": 4, "g": 1.65, "alpha_in": 0.50, "input_parity": "even", "target_alpha": 0.74, "target_gamma": 0.076, "target_parity": "even", "paper_p_trial": 3.85e-7, "paper_fidelity": 1.000},
    {"label": "Table III ECS (4,4), alpha=1.00", "m": 4, "n": 4, "g": 1.65, "alpha_in": 1.00, "input_parity": "even", "target_alpha": 1.53, "target_gamma": 0.309, "target_parity": "even", "paper_p_trial": 6.41e-7, "paper_fidelity": 0.999},
    {"label": "Table III ECS (4,4), alpha=1.50", "m": 4, "n": 4, "g": 1.65, "alpha_in": 1.50, "input_parity": "even", "target_alpha": 2.63, "target_gamma": 0.688, "target_parity": "even", "paper_p_trial": 8.84e-7, "paper_fidelity": 0.984},

    # Additional Table III row supplied after the original table calculation.
    {"label": "Table III ECS (6,6), alpha=1.20", "m": 6, "n": 6, "g": 1.42, "alpha_in": 1.20, "input_parity": "even", "target_alpha": 1.85, "target_gamma": 0.438, "target_parity": "even", "paper_fidelity": 0.999},

    # Table IV: non-catalytic configurations
    {"label": "Table IV ECS (1,2), alpha=1.18", "m": 1, "n": 2, "g": 1.05, "alpha_in": 1.18, "input_parity": "even", "target_alpha": 2.02, "target_gamma": 0.23, "target_parity": "odd", "paper_p_trial": 1.74e-2, "paper_fidelity": 0.996},
    {"label": "Table IV ECS (1,3), alpha=1.97", "m": 1, "n": 3, "g": 1.01, "alpha_in": 1.97, "input_parity": "even", "target_alpha": 3.45, "target_gamma": 0.26, "target_parity": "even", "paper_p_trial": 1.13e-3, "paper_fidelity": 0.991},
    {"label": "Table IV ECS (0,2), alpha=1.97", "m": 0, "n": 2, "g": 1.01, "alpha_in": 1.97, "input_parity": "even", "target_alpha": 3.46, "target_gamma": 0.26, "target_parity": "even", "paper_p_trial": 4.59e-3, "paper_fidelity": 0.991},
    {"label": "Table IV ECS (3,2), alpha=1.45", "m": 3, "n": 2, "g": 1.60, "alpha_in": 1.45, "input_parity": "even", "target_alpha": 2.00, "target_gamma": 0.49, "target_parity": "odd", "paper_p_trial": 1.34e-5, "paper_fidelity": 0.996},
    {"label": "Table IV ECS (1,6), alpha=2.50", "m": 1, "n": 6, "g": 1.10, "alpha_in": 2.50, "input_parity": "even", "target_alpha": 5.23, "target_gamma": 0.39, "target_parity": "odd", "paper_fidelity": 0.988},
    {"label": "Table IV OCS (1,2), alpha=1.97", "m": 1, "n": 2, "g": 1.05, "alpha_in": 1.97, "input_parity": "odd", "target_alpha": 2.69, "target_gamma": 0.20, "target_parity": "even", "paper_p_trial": 2.20e-2, "paper_fidelity": 0.991},
    {"label": "Table IV OCS (0,2), alpha=1.97", "m": 0, "n": 2, "g": 1.01, "alpha_in": 1.97, "input_parity": "odd", "target_alpha": 3.46, "target_gamma": 0.25, "target_parity": "odd", "paper_p_trial": 4.59e-3, "paper_fidelity": 0.991},
    {"label": "Table IV OCS (3,2), alpha=1.76", "m": 3, "n": 2, "g": 3.30, "alpha_in": 1.76, "input_parity": "odd", "target_alpha": 2.31, "target_gamma": 0.35, "target_parity": "even", "paper_p_trial": 5.52e-6, "paper_fidelity": 0.993},
    {"label": "Table IV OCS (4,2), alpha=1.34", "m": 4, "n": 2, "g": 1.70, "alpha_in": 1.34, "input_parity": "odd", "target_alpha": 2.15, "target_gamma": 0.51, "target_parity": "odd", "paper_p_trial": 2.59e-7, "paper_fidelity": 0.993},
    {"label": "Table IV OCS (1,7), alpha=2.50", "m": 1, "n": 7, "g": 1.10, "alpha_in": 2.50, "input_parity": "odd", "target_alpha": 5.68, "target_gamma": 0.42, "target_parity": "odd", "paper_fidelity": 0.987},

    # Table I: cubic phase states (CPS), coherent-state input
    # alpha_in and correction_displacement are purely imaginary, as in Table I.
    {"label": "Table I CPS (0,4)", "family": "cps", "m": 0, "n": 4, "g": 1.46, "alpha_in": -2.145j, "target_cubicity": 0.103, "target_squeezing": 0.00, "correction_squeezing": 0.521, "correction_displacement": 2.963j, "paper_p_trial": 8.28e-2, "paper_fidelity": 0.997},
    {"label": "Table I CPS (1,1)", "family": "cps", "m": 1, "n": 1, "g": 1.08, "alpha_in": -4.136j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.203, "correction_displacement": 4.406j, "paper_p_trial": 1.17e-2, "paper_fidelity": 0.975},
    {"label": "Table I CPS (1,4)", "family": "cps", "m": 1, "n": 4, "g": 1.15, "alpha_in": -1.448j, "target_cubicity": 0.146, "target_squeezing": -0.01, "correction_squeezing": 0.589, "correction_displacement": 2.493j, "paper_p_trial": 8.82e-3, "paper_fidelity": 0.989},
    {"label": "Table I CPS (1,5)", "family": "cps", "m": 1, "n": 5, "g": 1.07, "alpha_in": -2.260j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.469, "correction_displacement": 3.401j, "paper_p_trial": 3.75e-4, "paper_fidelity": 0.991},
    {"label": "Table I CPS (2,2)", "family": "cps", "m": 2, "n": 2, "g": 2.06, "alpha_in": -3.676j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.399, "correction_displacement": 2.786j, "paper_p_trial": 1.13e-5, "paper_fidelity": 0.993},
    {"label": "Table I CPS (2,5)", "family": "cps", "m": 2, "n": 5, "g": 1.05, "alpha_in": -1.614j, "target_cubicity": 0.100, "target_squeezing": 0.00, "correction_squeezing": 0.504, "correction_displacement": 2.703j, "paper_p_trial": 3.80e-4, "paper_fidelity": 0.995},
    {"label": "Table I CPS (3,3)", "family": "cps", "m": 3, "n": 3, "g": 2.34, "alpha_in": -3.933j, "target_cubicity": 0.128, "target_squeezing": 0.00, "correction_squeezing": 0.527, "correction_displacement": 3.020j, "paper_p_trial": 6.69e-7, "paper_fidelity": 0.992},
    {"label": "Table I CPS (4,4)", "family": "cps", "m": 4, "n": 4, "g": 2.47, "alpha_in": -3.960j, "target_cubicity": 0.166, "target_squeezing": -0.01, "correction_squeezing": 0.647, "correction_displacement": 3.258j, "paper_p_trial": 1.19e-7, "paper_fidelity": 0.987},
]

print(f"tanh(rho_prime) = {math.sqrt(1.0 - 1.0 / G_PREP**2):.8f}")
print(f"ETA_PREP={ETA_PREP:.2f}, ETA_OUT={ETA_OUT:.2f}, N_DIM={N_DIM}, J_MAX={J_MAX}, L_MAX={L_MAX}")
print(f"Configured cases: {len(CASES)}; TOP_CHANNELS_PER_CASE={TOP_CHANNELS_PER_CASE}")


tanh(rho_prime) = 0.30004387
ETA_PREP=0.95, ETA_OUT=0.95, N_DIM=50, J_MAX=8, L_MAX=14
Configured cases: 44; TOP_CHANNELS_PER_CASE=0


## 论文的态与条件概率

`opa_heralded_state(signal, j, l, gain)` 返回第二篇论文式 (3)-(4) 的未归一化信号态 $|\Psi_{\psi,j,l}\rangle$。其范数平方就是

$$
P_{\psi}(g,j,l)=\|\Psi_{\psi,j,l}\|^2.
$$

后续对目标态施加的高斯校正是幺正操作，不改变这一成功概率；它只影响保真度的参照目标。对 CPS，目标态遵循论文式 (13)，并使用 Table I 的 $S(r_{\rm corr})D(\beta_{\rm corr})$ 校正。

In [3]:
# 3. 第二篇论文式 (3)、(4)、(6)、(13)、(18)
def cat_state(dim: int, alpha: float, parity: str) -> qt.Qobj:
    if parity not in {"odd", "even"}:
        raise ValueError("parity must be 'odd' or 'even'.")
    sign = -1.0 if parity == "odd" else 1.0
    return (qt.coherent(dim, alpha) + sign * qt.coherent(dim, -alpha)).unit()


def attenuate_fock_amplitudes(state: qt.Qobj, gain: float) -> qt.Qobj:
    amplitudes = state.full().ravel()
    factors = gain ** (-np.arange(len(amplitudes)))
    return qt.Qobj((amplitudes * factors).reshape((-1, 1)), dims=state.dims)


def h_coefficient(k: int, m: int, n: int, gain: float) -> float:
    added = n - m + k
    if added < 0:
        return 0.0
    G = math.sqrt(1.0 - 1.0 / gain**2)
    numerator = ((-1) ** added) * G ** (n - m + 2 * k)
    numerator *= math.sqrt(math.factorial(n) * math.factorial(m))
    denominator = gain ** (m - k + 1)
    denominator *= math.factorial(k) * math.factorial(added) * math.factorial(m - k)
    return numerator / denominator


def opa_heralded_state(signal: qt.Qobj, m_actual: int, n_actual: int, gain: float) -> qt.Qobj:
    """Second-paper Eq. (3): unnormalized conditional signal state."""
    if gain < 1.0:
        raise ValueError("OPA gain must satisfy g >= 1.")
    a = qt.destroy(signal.shape[0])
    output = 0.0 * signal
    for k in range(m_actual + 1):
        added = n_actual - m_actual + k
        coefficient = h_coefficient(k, m_actual, n_actual, gain)
        if coefficient != 0.0:
            output += coefficient * (a.dag() ** added) * attenuate_fock_amplitudes((a ** k) * signal, gain)
    return output


def squeezed_cat_target(dim: int, alpha: float, gamma: float, parity: str) -> qt.Qobj:
    return (qt.squeeze(dim, gamma) * cat_state(dim, alpha, parity)).unit()


def cubic_phase_target(dim: int, cubicity: float, squeezing: float) -> qt.Qobj:
    """Table-I CPS target from Eq. (13), using the manuscript's existing numerical convention x=a+a^dagger."""
    a = qt.destroy(dim)
    x = a + a.dag()
    cubic_phase = (1j * cubicity * (x ** 3)).expm()
    return (cubic_phase * qt.squeeze(dim, squeezing) * qt.basis(dim, 0)).unit()


def gaussian_correction_operator(dim: int, squeezing: float, displacement: complex) -> qt.Qobj:
    """Table-I correction S(r_corr) D(beta_corr), matching the original CPS calculation code."""
    return qt.squeeze(dim, squeezing) * qt.displace(dim, displacement)


## 两级有限效率 PNRD 模型

若真实有 $q$ 个光子而 detector 登记为 $r$ 个，二项 PNRD 权重是

$$
w_\eta(q\to r)=\binom{q}{r}\eta^r(1-\eta)^{q-r}.
$$

完整函数对真实 $(j,l)$ 通道求和，并返回未归一化条件密度矩阵。取其迹得到原始注册概率；归一化后才能计算有限效率保真度。

In [4]:
# 4. 成功概率与有限效率混态：第一篇修订稿式 (46) 和式 (51)
def fock_preparation_probability(j_actual: int, g_prep: float) -> float:
    """Auxiliary-SPDC probability P_Fock(j,g_prime)."""
    return (1.0 / g_prep**2) * (1.0 - 1.0 / g_prep**2) ** j_actual


def pnrd_weight(true_count: int, registered_count: int, efficiency: float) -> float:
    """Bernoulli PNRD weight: true_count photons are registered as registered_count."""
    if true_count < registered_count:
        return 0.0
    return (
        math.comb(true_count, registered_count)
        * efficiency**registered_count
        * (1.0 - efficiency) ** (true_count - registered_count)
    ) 


def registered_fock_preparation_probability(registered_count: int, g_prep: float, efficiency: float, true_max: int) -> float:
    """Finite-efficiency auxiliary-SPDC preparation probability in Eq. (50)."""
    return sum(
        pnrd_weight(j_actual, registered_count, efficiency)
        * fock_preparation_probability(j_actual, g_prep)
        for j_actual in range(registered_count, true_max + 1)
    )


def real_scalar(value) -> float:
    return float(np.real_if_close(value))


def evaluate_case(case: dict, eta_prep: float, eta_out: float, j_max: int, l_max: int) -> tuple[dict, list[dict]]:
    """Evaluate Eq. (46), Eq. (51), conditional fidelity, and registered-channel contributions."""
    m, n, gain = case["m"], case["n"], case["g"]
    family = case.get("family", "cat")
    if family == "cps":
        signal = qt.coherent(N_DIM, case["alpha_in"])
        target = cubic_phase_target(N_DIM, case["target_cubicity"], case["target_squeezing"])
        correction = gaussian_correction_operator(
            N_DIM, case["correction_squeezing"], case["correction_displacement"]
        )
    elif family == "cat":
        signal = cat_state(N_DIM, case["alpha_in"], case["input_parity"])
        target = squeezed_cat_target(N_DIM, case["target_alpha"], case["target_gamma"], case["target_parity"])
        correction = qt.qeye(N_DIM)
    else:
        raise ValueError(f"Unknown state family: {family}")

    state_cache: dict[tuple[int, int], qt.Qobj] = {}

    def conditional_state(j_actual: int, l_actual: int) -> qt.Qobj:
        key = (j_actual, l_actual)
        if key not in state_cache:
            state_cache[key] = opa_heralded_state(signal, j_actual, l_actual, gain)
        return state_cache[key]

    ideal_ket = conditional_state(m, n)
    p_psi_mn = real_scalar(ideal_ket.overlap(ideal_ket))
    p_fock_m = fock_preparation_probability(m, G_PREP)
    p_fock_m_eta_prep = registered_fock_preparation_probability(m, G_PREP, eta_prep, j_max)

    # Revised-paper Eq. (46): retain only the correct-registration path j=m, l=n.
    p_dominant_eq46 = eta_prep**m * eta_out**n * p_fock_m * p_psi_mn

    # Kept only to audit the old Tables I-IV convention. Do not use as a corrected success probability.
    p_legacy_single_eta = eta_out * p_fock_m * p_psi_mn

    corrected_ideal_ket = correction * ideal_ket
    fidelity_ideal_table_parameters = abs(target.overlap(corrected_ideal_ket.unit())) ** 2

    rho_unnormalized = 0.0 * qt.ket2dm(qt.basis(N_DIM, 0))
    channel_rows: list[dict] = []
    for j_actual in range(m, j_max + 1):
        prep_weight = pnrd_weight(j_actual, m, eta_prep)
        prep_factor = prep_weight * fock_preparation_probability(j_actual, G_PREP)
        if prep_factor == 0.0:
            continue
        for l_actual in range(n, l_max + 1):
            out_weight = pnrd_weight(l_actual, n, eta_out)
            if out_weight == 0.0:
                continue
            ket = conditional_state(j_actual, l_actual)
            p_psi_jl = real_scalar(ket.overlap(ket))
            contribution = prep_factor * out_weight * p_psi_jl
            if contribution == 0.0:
                continue
            corrected_ket = correction * ket
            rho_unnormalized += prep_factor * out_weight * qt.ket2dm(corrected_ket)
            channel_rows.append({
                "label": case["label"], "m_registered": m, "n_registered": n,
                "j_actual": j_actual, "l_actual": l_actual,
                "prep_weight": prep_weight,
                "P_fock_j": fock_preparation_probability(j_actual, G_PREP),
                "out_weight": out_weight, "P_psi_j_l": p_psi_jl,
                "contribution": contribution,
                "is_correct_registration": (j_actual == m and l_actual == n),
            })

    # Revised-paper Eq. (51): sum every true (j,l) channel that can be registered as (m,n).
    p_trial_eq51 = real_scalar(rho_unnormalized.tr())
    if p_trial_eq51 <= 0.0:
        raise RuntimeError("Registered-event probability is zero; increase cutoffs or check parameters.")
    rho_conditioned = rho_unnormalized / p_trial_eq51
    fidelity_eta = real_scalar(target.dag() * rho_conditioned * target)

    for row in channel_rows:
        row["fraction_of_registered_events"] = row["contribution"] / p_trial_eq51
    channel_rows.sort(key=lambda row: row["contribution"], reverse=True)

    correct_row = next(row for row in channel_rows if row["is_correct_registration"])
    if not np.isclose(correct_row["contribution"], p_dominant_eq46, rtol=1e-12, atol=1e-15):
        raise AssertionError(f"Eq. (46) dominant-path check failed for {case['label']}.")

    paper_p_trial_old = case.get("paper_p_trial", np.nan)
    return {
        "label": case["label"], "family": family, "m": m, "n": n, "g": gain,
        "alpha_in": case["alpha_in"], "input_parity": case.get("input_parity", "coherent"),
        "eta_prep": eta_prep, "eta_out": eta_out,
        "P_fock_m": p_fock_m,
        "P_fock_m_eta_prep": p_fock_m_eta_prep,
        "P_fock_n_requested": fock_preparation_probability(n, G_PREP),
        "P_psi_mn": p_psi_mn,
        "P_dominant_eq46": p_dominant_eq46,
        "P_trial_eq51": p_trial_eq51,
        "P_legacy_single_eta": p_legacy_single_eta,
        "F_ideal_table_parameters": float(fidelity_ideal_table_parameters),
        "F_conditional_eta": fidelity_eta,
        "correct_registration_fraction": correct_row["fraction_of_registered_events"],
        "paper_P_trial_old": paper_p_trial_old,
        "legacy_to_old_table_ratio": p_legacy_single_eta / paper_p_trial_old if np.isfinite(paper_p_trial_old) else np.nan,
        "eq51_to_old_table_ratio": p_trial_eq51 / paper_p_trial_old if np.isfinite(paper_p_trial_old) else np.nan,
        "paper_F_ideal": case.get("paper_fidelity", np.nan),
    }, channel_rows


In [5]:
# 5. 运行计算和 Eq. (46)/(51) 一致性检查
summary_rows, contribution_rows = [], []

for case in CASES:
    summary, contributions = evaluate_case(case, ETA_PREP, ETA_OUT, J_MAX, L_MAX)
    summary_rows.append(summary)
    contribution_rows.extend(contributions)

    # Perfect-detector limit: Eq. (51) reduces to P_Fock(m) P_psi(m,n).
    perfect, _ = evaluate_case(case, 1.0, 1.0, case["m"], case["n"])
    expected_perfect = summary["P_fock_m"] * summary["P_psi_mn"]
    if not np.isclose(perfect["P_trial_eq51"], expected_perfect, rtol=1e-12, atol=1e-14):
        raise AssertionError(f"Perfect-detector limit failed for {case['label']}.")

print("=" * 180)
print("Second-paper success probabilities with revised-paper Eq. (46) and Eq. (51)")
print("=" * 180)
print(
    f"{'case':<36} {'P_eq46 dominant':>17} {'P_eq51 total':>17} "
    f"{'old single-eta':>17} {'F_eta':>10} {'correct path':>14} {'Eq51/old':>11}"
)
print("-" * 180)
for row in summary_rows:
    print(
        f"{row['label']:<36} {row['P_dominant_eq46']:>17.5e} {row['P_trial_eq51']:>17.5e} "
        f"{row['P_legacy_single_eta']:>17.5e} {row['F_conditional_eta']:>10.6f} "
        f"{row['correct_registration_fraction']:>13.2%} {row['eq51_to_old_table_ratio']:>11.5f}"
    )

print("\nColumn meanings:")
print("  P_eq46 dominant: correct-registration (j,l)=(m,n) term, with eta_prep^m eta_out^n.")
print("  P_eq51 total: complete finite-efficiency registered-event probability; use this to update the paper tables.")
print("  old single-eta: legacy eta_out P_Fock(m) P_psi(m,n), printed only for audit.")

# Table V's preparation column is P_Fock^(eta_prep)(m), not P_Fock(n).
# n labels the OPA-output heralding count; its ideal P_Fock(n) is printed only as a reference.
print("\nAuxiliary-SPDC Fock-state preparation probabilities:")
print("Table-V quantity: P_Fock^(eta_prep)(m); P_Fock(n) is a reference only.")
print(f"{'case':<44} {'m':>3} {'n':>3} {'P_Fock^eta(m)':>18} {'P_Fock(n), ref.':>18}")
print("-" * 92)
for row in summary_rows:
    print(
        f"{row['label']:<44} {row['m']:>3d} {row['n']:>3d} "
        f"{row['P_fock_m_eta_prep']:>18.8e} {row['P_fock_n_requested']:>18.8e}"
    )


Second-paper success probabilities with revised-paper Eq. (46) and Eq. (51)
case                                   P_eq46 dominant      P_eq51 total    old single-eta      F_eta   correct path    Eq51/old
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Table II OCS (1,1), alpha=0.30             1.39240e-03       1.60453e-03       1.46568e-03   0.872215        86.78%     1.15434
Table II OCS (1,1), alpha=1.06             2.74625e-03       3.03476e-03       2.89079e-03   0.901815        90.49%     1.11163
Table II OCS (1,1), alpha=1.31             3.95466e-03       4.32087e-03       4.16280e-03   0.895215        91.52%     1.09667
Table II OCS (2,2), alpha=0.30             2.51347e-04       2.81675e-04       2.93159e-04   0.897132        89.23%     1.01322
Table II OCS (2,2), alpha=1.06             3.01386e-04       3.42424e-04       3.51521e-04   0.886213 

In [6]:
# 6. 保存 Eq. (51) 结果。修改 ETA 或截断后会覆盖当前 Eq. (51) 运行的 CSV。
summary_columns = list(summary_rows[0].keys())
with (OUTPUT_DIR / "summary_eq51.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=summary_columns)
    writer.writeheader()
    writer.writerows(summary_rows)

contribution_columns = list(contribution_rows[0].keys())
with (OUTPUT_DIR / "channel_contributions_eq51.csv").open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=contribution_columns)
    writer.writeheader()
    writer.writerows(contribution_rows)

print(f"\nSaved: {(OUTPUT_DIR / 'summary_eq51.csv').resolve()}")
print(f"Saved: {(OUTPUT_DIR / 'channel_contributions_eq51.csv').resolve()}")



Saved: C:\Users\10260\second code combination\outputs\second_paper_eq51_corrected_outputs\summary_eq51.csv
Saved: C:\Users\10260\second code combination\outputs\second_paper_eq51_corrected_outputs\channel_contributions_eq51.csv


In [7]:
# 7. Eq. (51) 截断收敛检查：比较当前截断与更大的截断。
CHECK_J_MAX = J_MAX + 2
CHECK_L_MAX = L_MAX + 4

print(f"Comparing Eq. (51) at (J_MAX,L_MAX)=({J_MAX},{L_MAX}) and ({CHECK_J_MAX},{CHECK_L_MAX})")
for case, base in zip(CASES, summary_rows):
    enlarged, _ = evaluate_case(case, ETA_PREP, ETA_OUT, CHECK_J_MAX, CHECK_L_MAX)
    probability_relative_change = abs(enlarged["P_trial_eq51"] - base["P_trial_eq51"]) / base["P_trial_eq51"]
    fidelity_absolute_change = abs(enlarged["F_conditional_eta"] - base["F_conditional_eta"])
    print(
        f"{case['label']}: relative change in P_eq51={probability_relative_change:.3e}, "
        f"absolute change in F_eta={fidelity_absolute_change:.3e}"
    )


Comparing Eq. (51) at (J_MAX,L_MAX)=(8,14) and (10,18)
Table II OCS (1,1), alpha=0.30: relative change in P_eq51=0.000e+00, absolute change in F_eta=0.000e+00
Table II OCS (1,1), alpha=1.06: relative change in P_eq51=0.000e+00, absolute change in F_eta=0.000e+00
Table II OCS (1,1), alpha=1.31: relative change in P_eq51=0.000e+00, absolute change in F_eta=0.000e+00
Table II OCS (2,2), alpha=0.30: relative change in P_eq51=0.000e+00, absolute change in F_eta=0.000e+00
Table II OCS (2,2), alpha=1.06: relative change in P_eq51=1.583e-16, absolute change in F_eta=4.441e-16
Table II OCS (2,2), alpha=1.31: relative change in P_eq51=2.804e-16, absolute change in F_eta=0.000e+00
Table II OCS (3,3), alpha=0.30: relative change in P_eq51=0.000e+00, absolute change in F_eta=0.000e+00
Table II OCS (3,3), alpha=1.06: relative change in P_eq51=6.297e-16, absolute change in F_eta=3.331e-16
Table II OCS (3,3), alpha=1.31: relative change in P_eq51=4.878e-15, absolute change in F_eta=3.331e-15
Table II 

## 如何使用修正后的结果

### 应报告哪一列

- `P_trial_eq51`：第一篇修订稿式 (51) 的完整有限效率成功概率。这是更新第二篇 Tables I--IV 成功概率时应使用的主列。
- `P_dominant_eq46`：第一篇修订稿式 (46) 的正确登记主导通道。它包含 $\eta_{\rm prep}^{m}\eta_{\rm out}^{n}$，可用于说明高阶 $(m,n)$ 为什么额外受效率惩罚。
- `P_legacy_single_eta`：旧代码的 $\eta_{\rm out}P_{\rm Fock}(m)P_\psi(m,n)$。它仅用于追溯旧表数值，不能继续作为修订后的理论预测。

### 为什么式 (51) 的理想通道给出效率的 m、n 次方

对正确登记通道 $j=m,l=n$，二项权重为

$$
w_{\eta_{\rm prep}}(m\to m)=\eta_{\rm prep}^m,
\qquad
w_{\eta_{\rm out}}(n\to n)=\eta_{\rm out}^n.
$$

因此式 (51) 中的这一项严格等于

$$
\eta_{\rm prep}^m\eta_{\rm out}^nP_{\rm Fock}(m,g')P_\psi(g,m,n),
$$

即式 (46)。只有在 $m=n=1$ 时，它才恰好有一个总的 $\eta_{\rm prep}\eta_{\rm out}$ 因子；一般不能简化为单个 $\eta$。

### 重要解释

式 (51) 的总注册概率可以大于或小于主导通道项，因为 $j>m$、$l>n$ 的漏检事件也会被登记为 $(m,n)$。这类事件提高的是“登记率”，却会使条件输出成为混态；因此需要同时报告 `F_conditional_eta` 和 `correct_registration_fraction`。
